In [ ]:
import arviz as az
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
from IPython.display import Markdown

from emu_renewal.inputs import ANALYSIS_TYPES, get_latest_analyses
from emu_renewal.outputs import get_output_dir, melt_df_except_first_level, get_multianalysis_dispvals_from_idatas, get_multianalysis_procvals_from_idatas, get_multianalysis_likelihoods
from emu_renewal.outputs import get_multianalysis_ind_spaghetti, get_multianalysis_procvals
from emu_renewal.plotting import plot_process_comparison, plot_updates_comparison, plot_mob_update_comparison

set_matplotlib_formats("svg")

In [ ]:
countries = ["France", "Sweden"]
analysis_times = {c: get_latest_analyses(c, ANALYSIS_TYPES) for c in countries}

In [ ]:
idatas = {}
proc_dfs = {}
disp_dfs = {}
for country in countries:
    country_analysis_times = analysis_times[country]
    country_idatas = {k: az.from_netcdf(get_output_dir(country, k, v) / "idata.nc") for k, v in country_analysis_times.items()}
    idatas[country] = country_idatas
    proc_dfs[country] = melt_df_except_first_level(get_multianalysis_procvals_from_idatas(country_idatas))
    disp_dfs[country] = melt_df_except_first_level(get_multianalysis_dispvals_from_idatas(country_idatas))

In [ ]:
n_countries = len(countries)
disp_fig, axes = plt.subplots(1, n_countries, figsize=(6 * n_countries, 6), sharex=True)
for c, country in enumerate(countries):
    c_ax = sns.kdeplot(disp_dfs[country], fill=True, ax=axes[c])
    c_ax.set_title(country)

In [ ]:
disp_fig.savefig("disp_fig.svg")

In [ ]:
#| fig-cap: "Comparison of likelihoods by inclusion of candidate mobility modifiers."
like_fig, ax = plt.subplots(figsize=(6, 6))
likelihoods = melt_df_except_first_level(get_multianalysis_likelihoods(country, analysis_times))
axes = sns.kdeplot(likelihoods, fill=True, ax=ax)
axes.set_title(country);

In [ ]:
#| fig-cap: "Variable process residual function by inclusion of candidate mobility modifiers."
spagh_df = get_multianalysis_ind_spaghetti(country, "process", all_analysis_times)
proc_fig = plot_process_comparison(spagh_df, all_analysis_times.keys(), ["k", "cornflowerblue", "mediumblue", "red", "darkred"], 0.1);
proc_fig.legend(bbox_to_anchor=[0.85, 0.75], loc="right")

In [ ]:
fig = axes.get_figure()

In [ ]:
fig.savefig(f"{country}_like_comparison.svg")

In [ ]:
#| tbl-cap: "Median likelihood values by inclusion of candidate mobility modifiers."
medians = likelihoods.median()
medians.index.name = "Analysis type"
medians.name = "Median values"
Markdown(medians.to_markdown())